# Transaction Anomaly Detection - Solidgate Hackathon

The dataset contains 1,000,000 transactions from 2025 with no `is_anomaly` label — this is an **unsupervised anomaly detection** problem. Anomalies must be found through statistical deviations, inconsistencies, and artificial patterns rather than learned from labels.

This is Part 1 of a multi-notebook analysis. It covers only the foundational understanding of the data — structure, integrity, baseline distributions. Each confirmed anomaly cluster gets its own notebook so the investigation for each one can be read (and re-run) independently:

- **01_eda.ipynb** (this notebook): data loading, integrity, baseline stats, datetime conversion, missing-value hypotheses.
- **02_anomaly_bank_id_777.ipynb**: the synthetic `bank_id == 777` cluster.
- **03_anomaly_latency_psp_gamma.ipynb**: the full-dataset latency distribution, hidden clusters, and the systematic category x time scan (psp_gamma incident).
- **04_anomaly_refund_psp_beta.ipynb**: the refund/over-refund investigation (psp_beta).
- **05_conclusions_next_steps.ipynb**: cross-notebook summary and the plan for remaining work.

## Contents
1. Data loading and structure
2. Data integrity: keys and missing values
3. Baseline statistics and category distributions
4. Datetime conversion
5. Missing-value hypotheses (crosstab)
6. Additional integrity invariants (cross-column checks)


## 1. Data loading and structure

In [1]:
import pandas as pd

df = pd.read_csv('/home/veronika/Anomaly_Hunter_Solidgate/hackathon_int20h_dataset_test.csv')

In [2]:
print(df.shape)
print(df.head(5))

(1000000, 18)
            created_at  order_id         processed_at order_type  user_id  \
0  2025-07-01 09:21:23         1  2025-07-01 09:21:32      first   692925   
1  2025-09-01 01:15:47         2  2025-09-01 01:15:57  recurring   452913   
2  2025-06-24 23:38:35         3  2025-06-24 23:38:39      first   784680   
3  2025-04-23 04:42:13         4  2025-04-23 04:42:21      first   300037   
4  2025-03-14 20:15:32         5  2025-03-14 20:15:42      first   996803   

  ip_country currency  amount payment_method order_payment_type bin_country  \
0        DEU      EUR    4.60      googlepay                NaN         GBR   
1        CAN      CAD   54.80           card          recurring         CAN   
2        USA      USD    9.99           card                NaN         USA   
3        CAN      CAD    1.37           card                NaN         CAN   
4        DEU      EUR    0.92           card                NaN         GBR   

   bank_id     psp_id  has_refund  refunded_amou

In [3]:
print(df.head().info())
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   created_at          5 non-null      object 
 1   order_id            5 non-null      int64  
 2   processed_at        5 non-null      object 
 3   order_type          5 non-null      object 
 4   user_id             5 non-null      int64  
 5   ip_country          5 non-null      object 
 6   currency            5 non-null      object 
 7   amount              5 non-null      float64
 8   payment_method      5 non-null      object 
 9   order_payment_type  1 non-null      object 
 10  bin_country         5 non-null      object 
 11  bank_id             5 non-null      int64  
 12  psp_id              5 non-null      object 
 13  has_refund          5 non-null      bool   
 14  refunded_amount     5 non-null      float64
 15  is_secured          5 non-null      bool   
 16  status      

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 18 columns):
 #   Column              Non-Null Count    Dtype  
---  ------              --------------    -----  
 0   created_at          1000000 non-null  object 
 1   order_id            1000000 non-null  int64  
 2   processed_at        1000000 non-null  object 
 3   order_type          1000000 non-null  object 
 4   user_id             1000000 non-null  int64  
 5   ip_country          1000000 non-null  object 
 6   currency            1000000 non-null  object 
 7   amount              1000000 non-null  float64
 8   payment_method      1000000 non-null  object 
 9   order_payment_type  599474 non-null   object 
 10  bin_country         1000000 non-null  object 
 11  bank_id             1000000 non-null  int64  
 12  psp_id              1000000 non-null  object 
 13  has_refund          1000000 non-null  bool   
 14  refunded_amount     1000000 non-null  float64
 15  is_secured      

In [4]:
print(df.nunique())

created_at             983310
order_id              1000000
processed_at           983301
order_type                  2
user_id                603337
ip_country                  8
currency                    7
amount                    112
payment_method              3
order_payment_type          4
bin_country                 8
bank_id                    50
psp_id                      3
has_refund                  2
refunded_amount           201
is_secured                  2
status                      2
error_code                 13
dtype: int64


**Finding:** `created_at` and `processed_at` load as `object`, even though they are timestamps — they need explicit conversion to `datetime` (done in section 4) before any time-based analysis is possible.

## 2. Data integrity: keys and missing values

In [5]:
print(df['order_id'].duplicated().sum())

0


In [6]:
print(df.isna().sum())

created_at                 0
order_id                   0
processed_at               0
order_type                 0
user_id                    0
ip_country                 0
currency                   0
amount                     0
payment_method             0
order_payment_type    400526
bin_country                0
bank_id                    0
psp_id                     0
has_refund                 0
refunded_amount            0
is_secured                 0
status                     0
error_code            525114
dtype: int64


**Observation:** no duplicate `order_id` values - the key is intact. Two large blocks of missing values stand out: `order_payment_type` (~400k) and `error_code` (~525k). Hypothesis: both are explained by structural logic rather than data quality issues (tested with crosstabs in section 5).

## 3. Baseline statistics and category distributions

In [7]:
print(df.describe())

             order_id         user_id          amount         bank_id  \
count  1000000.000000  1000000.000000  1000000.000000  1000000.000000   
mean    500000.500000   550083.804859      115.133693       25.474309   
std     288675.278933   259853.867901      390.972249       23.637364   
min          1.000000   100002.000000        0.780000        1.000000   
25%     250000.750000   324983.000000        9.990000       13.000000   
50%     500000.500000   550019.000000       20.100000       25.000000   
75%     750000.250000   775152.500000       50.000000       37.000000   
max    1000000.000000   999997.000000     8240.000000      777.000000   

       refunded_amount     error_code  
count   1000000.000000  474886.000000  
mean          2.339364       2.905473  
std          43.627719       0.497555  
min           0.000000       0.010000  
25%           0.000000       3.020000  
50%           0.000000       3.020000  
75%           0.000000       3.080000  
max        6190.000000

Next: `value_counts()` across every column, hunting for rare categories - values whose frequency or format stands out from the rest.

In [8]:
for col in df.columns:
    print(df[col].value_counts())
    print("\n")

created_at
2025-03-17 16:23:43    4
2025-10-06 16:13:36    3
2025-08-07 20:02:12    3
2025-12-25 22:17:05    3
2025-07-04 10:30:32    3
                      ..
2025-03-09 15:57:53    1
2025-12-01 12:16:55    1
2025-12-26 08:35:56    1
2025-12-29 15:12:01    1
2025-12-01 11:20:19    1
Name: count, Length: 983310, dtype: int64


order_id
1          1
666658     1
666660     1
666661     1
666662     1
          ..
333338     1
333339     1
333340     1
333341     1
1000000    1
Name: count, Length: 1000000, dtype: int64




processed_at
2025-06-20 23:35:28    4
2025-11-23 15:41:42    4
2025-09-26 23:12:47    4
2025-02-07 02:41:06    4
2025-11-14 00:24:20    3
                      ..
2025-06-03 22:00:40    1
2025-11-12 08:01:12    1
2025-04-17 09:08:00    1
2025-01-22 19:21:51    1
2025-12-01 11:20:24    1
Name: count, Length: 983301, dtype: int64


order_type
recurring    599474
first        400526
Name: count, dtype: int64


user_id
164513    10
703576     9
923569     9
221556     9
673750     9
          ..
848567     1
533365     1
605817     1
199472     1
468046     1
Name: count, Length: 603337, dtype: int64




ip_country
USA    500471
CAN    150032
UKR     60233
GBR     60159
FRA     59892
DEU     59767
POL     59636
MEX     49810
Name: count, dtype: int64


currency
USD    500471
CAD    150032
EUR    119659
UAH     60233
GBP     60159
PLN     59636
MXN     49810
Name: count, dtype: int64


amount
20.0      81061
30.0      59771
1.0       59728
5.0       50048
40.0      45174
          ...  
8240.0      352
804.0       334
2775.0      333
603.0       326
3700.0      300
Name: count, Length: 112, dtype: int64


payment_method
card         700600
googlepay    149830
applepay     149570
Name: count, dtype: int64


order_payment_type
retry        150305
1-click      150196
rebill       149670
recurring    149303
Name: count, dtype: int64


bin_country
USA    438294
CAN    146130
UKR     70981
GBR     70789
FRA     70623
DEU     70487
POL     70468
MEX     62228
Name: count, dtype: int64


bank_id
42     20757
7      20746
31     20708
24     20691
1      20687
13     20599
21     20571
40     20

psp_id
psp_alpha    334013
psp_beta     333027
psp_gamma    332960
Name: count, dtype: int64


has_refund
False    962986
True      37014
Name: count, dtype: int64


refunded_amount
0.0       962986
10.0        2694
5.0         2385
15.0        2212
0.5         2093
           ...  
4954.0         2
2785.0         1
814.0          1
1490.0         1
284.0          1
Name: count, Length: 201, dtype: int64


is_secured
False    957663
True      42337
Name: count, dtype: int64


status
success    525114
fail       474886
Name: count, dtype: int64


error_code
3.02    150121
3.08     90293
3.10     76797
2.12     44700
2.01     43696
3.04     42896
4.09     17647
4.03      1520
3.01      1468
0.01      1468
5.01      1456
3.05      1442
2.03      1382
Name: count, dtype: int64




**Key finding:** `bank_id == 777` has only 635 transactions, and the ID itself is three digits while every other `bank_id` is one or two digits - an unusual format, a strong candidate for an artificially inserted group. Investigated in detail in section 6.

No other column (`psp_id`, other `bank_id` values, `currency`, `error_code`) shows a similar anomaly by frequency.

## 4. Datetime conversion

In [9]:
df['created_at'] = pd.to_datetime(df['created_at'])
df['processed_at'] = pd.to_datetime(df['processed_at'])

In [10]:
df.head()

,created_at,order_id,processed_at,order_type,user_id,ip_country,currency,amount,payment_method,order_payment_type,bin_country,bank_id,psp_id,has_refund,refunded_amount,is_secured,status,error_code
0,2025-07-01 09:21:23,1,2025-07-01 09:21:32,first,692925,DEU,EUR,4.60,googlepay,NaN,GBR,32,psp_alpha,False,0.0,False,fail,3.02
1,2025-09-01 01:15:47,2,2025-09-01 01:15:57,recurring,452913,CAN,CAD,54.80,card,recurring,CAN,1,psp_alpha,False,0.0,False,success,NaN
2,2025-06-24 23:38:35,3,2025-06-24 23:38:39,first,784680,USA,USD,9.99,card,NaN,USA,32,psp_alpha,False,0.0,False,fail,2.01
3,2025-04-23 04:42:13,4,2025-04-23 04:42:21,first,300037,CAN,CAD,1.37,card,NaN,CAN,31,psp_gamma,False,0.0,False,fail,3.04
4,2025-03-14 20:15:32,5,2025-03-14 20:15:42,first,996803,DEU,EUR,0.92,card,NaN,GBR,39,psp_beta,False,0.0,False,fail,2.12


## 5. Missing-value hypotheses (crosstab)

In [11]:
pd.crosstab(df['order_payment_type'], df['order_type'], margins=True, normalize='index')

order_type,recurring
order_payment_type,
1-click,1.0
rebill,1.0
recurring,1.0
retry,1.0
All,1.0


**Confirmed:** `order_payment_type` is populated exclusively for `order_type == recurring` (share of 1.0 for every value); it is always NaN for `first`. The missing values are a direct consequence of order type, not a data defect.

In [12]:
pd.crosstab(df['error_code'], df['status'], margins=True, normalize='index')

status,fail
error_code,
0.01,1.0
2.01,1.0
2.03,1.0
2.12,1.0
3.01,1.0
3.02,1.0
3.04,1.0
3.05,1.0
3.08,1.0


**Confirmed:** `error_code` is populated exactly when `status == fail` - expected, since an error code only makes sense for a failed transaction.

## 6. Additional integrity invariants (cross-column checks)

The crosstab checks in section 5 rely on `normalize='index'` proportions, which round to `1.0` even if a handful of exceptions exist in a column of hundreds of thousands of rows. The two relationships already found there, plus a few more business-logic rules, are re-verified below with **exact violation counts** instead of percentages - a systematic pass over rules of the form *"if column A has value X, column B must satisfy Y, with zero exceptions"*, rather than relying on distribution shape alone.

**Rule:** a transaction cannot be processed before it was created (`processed_at >= created_at` for every row).

In [13]:
inverse_latency = df['processed_at'] < df['created_at']
print("Rows with processed_at before created_at:", inverse_latency.sum())

Rows with processed_at before created_at: 0


**Confirmed:** zero exceptions - timestamp ordering holds for all 1,000,000 rows.

**Rule:** the `has_refund` flag must agree with `refunded_amount` in both directions - `has_refund == False` implies `refunded_amount == 0`, and `refunded_amount > 0` implies `has_refund == True`.

In [14]:
flagged_false_amount_positive = (~df['has_refund']) & (df['refunded_amount'] > 0)
flagged_true_amount_zero = df['has_refund'] & (df['refunded_amount'] == 0)
print("has_refund=False but refunded_amount>0:", flagged_false_amount_positive.sum())
print("has_refund=True but refunded_amount==0:", flagged_true_amount_zero.sum())

has_refund=False but refunded_amount>0: 0
has_refund=True but refunded_amount==0: 0


**Confirmed:** the flag and the amount never contradict each other, in either direction.

**Rule (re-verification of the section 5 crosstabs, with exact counts instead of proportions):** `order_payment_type` must be null whenever `order_type != 'recurring'`, and `error_code` must be null whenever `status == 'success'`.

In [15]:
non_recurring_with_payment_type = (df['order_type'] != 'recurring') & (~df['order_payment_type'].isnull())
error_code_on_success = (~df['error_code'].isnull()) & (df['status'] == 'success')
print("Non-recurring order with a payment type set:", non_recurring_with_payment_type.sum())
print("error_code present on a successful transaction:", error_code_on_success.sum())

Non-recurring order with a payment type set: 0
error_code present on a successful transaction: 0


**Confirmed:** the exact counts match the crosstab conclusion exactly - both relationships hold for every single row, not just "to four decimal places".

**Rule:** `refunded_amount` should never exceed `amount` - a merchant cannot refund more than was charged.

In [16]:
over_refunded = df['refunded_amount'] > df['amount']
print("Rows where refunded_amount > amount:", over_refunded.sum())

Rows where refunded_amount > amount: 2691


**Violation found - but not a new cluster.** These 2,691 rows are the same `psp_beta` over-refund anomaly investigated in depth in `04_anomaly_refund_psp_beta.ipynb` (5-9 August 2025, flat `+10` offset in every currency). This check confirms the cluster is exhaustive: there is no over-refund occurrence anywhere else in the year outside that window.